### Change data type of date columns from timestamp to date and save modified table to Silver layer 

In [0]:
from pyspark.sql.functions import *  
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger("landing_to_bronze")
catalog='catalog_dev'
source_schema='bronze'
target_schema='silver'
storage_account='onprimtocloudstorgaeacc'

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {source_schema}")

DataFrame[]

Setup Production logging and config

In [0]:
# function to create dataframe

def create_DF(table):
    df=spark.table(f'{catalog}.{source_schema}.{table}')
    logger.info(f"Starting ingestion for table: {catalog}.{source_schema}.{table} ")
    return df

In [0]:
# function to modify all columns having datatype timestamp to datatype date and 'yyyy-MM-dd' format

def modify_column(df,columns):
    modified_df=df
    for column_name in columns:
        logger.info(f"Modifying {column_name} ")
        modified_df=modified_df.withColumn(column_name, date_format(col(column_name),'yyyy-MM-dd').cast('date'))
    return modified_df

In [0]:
# function to save dataframe as table in Unity Catalog

def save_to_silver(df, table):
    logger.info(f"Saving {table}")
    target_path=f"abfss://{target_schema}@{storage_account}.dfs.core.windows.net/{table}"
    df.write.format('delta')\
        .mode('overwrite')\
        .option('path',target_path)\
        .option("overwriteSchema", "true")\
        .saveAsTable(f'{catalog}.{target_schema}.{table}')
    

In [0]:

def main():
    matrix =spark.sql(f'''SELECT table_name
    FROM {catalog}.information_schema.tables
    WHERE table_schema="{source_schema}"''')

    for row in matrix.collect():
        table_name=row.table_name
        logger.info(f'-----------Modifying table : {table_name}-----------')
        df=create_DF(table_name)  
        columns=[]
        for column in df.columns:
            if 'Date' in column:
                columns.append(column)
        
        modified_df=modify_column(df,columns)
        save_to_silver(modified_df,table_name)

    logger.info(f"Modification completed")
    logger.info(f"_______________________________________________________________________________________")


In [0]:

if __name__ == "__main__":
    main()

2026-06-03 10:36:06,525 - INFO - Received command c on object id p0
2026-06-03 10:36:07,425 - INFO - Starting ingestion for table: catalog_dev.bronze.address_customer 


Modifying table : address_customer


2026-06-03 10:36:08,447 - INFO - Modifying ModifiedDate 
2026-06-03 10:36:08,448 - INFO - Saving address_customer
2026-06-03 10:36:16,076 - INFO - Starting ingestion for table: catalog_dev.bronze.salesorderdetail 


Modifying table : salesorderdetail


2026-06-03 10:36:16,644 - INFO - Modifying ModifiedDate 
2026-06-03 10:36:16,645 - INFO - Saving salesorderdetail
2026-06-03 10:36:23,385 - INFO - Starting ingestion for table: catalog_dev.bronze.productdescription 


...

2026-06-03 10:37:05,863 - INFO - Modifying SellStartDate 
2026-06-03 10:37:05,863 - INFO - Modifying SellEndDate 
2026-06-03 10:37:05,864 - INFO - Modifying DiscontinuedDate 
2026-06-03 10:37:05,865 - INFO - Modifying ModifiedDate 
2026-06-03 10:37:05,866 - INFO - Saving product
2026-06-03 10:37:12,798 - INFO - Starting ingestion for table: catalog_dev.bronze.productcategory 


Modifying table : productcategory


2026-06-03 10:37:13,387 - INFO - Modifying ModifiedDate 
2026-06-03 10:37:13,388 - INFO - Saving productcategory
2026-06-03 10:37:19,179 - INFO - Starting ingestion for table: catalog_dev.bronze.address 


Modifying table : address


2026-06-03 10:37:19,759 - INFO - Modifying ModifiedDate 
2026-06-03 10:37:19,760 - INFO - Saving address
